# Sets

Split from `01_dictionaries_and_sets.ipynb` to keep this topic self-contained.

**Navigation:** [Previous: Dictionaries](./01_dictionaries.ipynb) · [Topic overview](./01_dictionaries_and_sets.ipynb)


<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
These patterns are used to build reusable data-processing scripts for FASTA/VCF/TSV handling, QC summaries, and automation glue code.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Mutable vs immutable objects and how that affects function behavior.
- Vectorized/dataframe workflows vs loops: both are useful, but for different bottlenecks.
- Reading errors: most failures come from dtype mismatches and unexpected missing values.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


# Module 9: Dictionaries and Sets

---

### Learning Objectives
- Master dictionary creation, access, methods, and iteration
- Use nested dictionaries and specialized collections (`defaultdict`, `Counter`)
- Understand set operations: union, intersection, difference, and frozenset
- Apply these structures to biological problems: codon tables, gene annotations, k-mer analysis

---

---

[← Previous: Module 8: Lists and Tuples](../../Tier_1_Python_for_Bioinformatics/08_Lists_and_Tuples/01_lists_and_tuples.ipynb) | [Next: Module 10: Comprehensions →](../../Tier_1_Python_for_Bioinformatics/10_Comprehensions/01_comprehensions.ipynb)

---

## 1. Dictionaries -- Key-Value Mapping

A dictionary maps **keys** to **values**. Looking up a key is extremely fast (O(1) on average), making dictionaries ideal for any kind of lookup table.

```
+---------------------------------------------------------+
|                 DICTIONARY STRUCTURE                     |
+---------------------------------------------------------+
|                                                         |
|   codon_table = {                                       |
|       "ATG": "M",     <-- key: value                    |
|       "TGG": "W",                                       |
|       "TAA": "*",                                       |
|   }                                                     |
|                                                         |
|   Access: codon_table["ATG"]  -->  "M"                  |
|                                                         |
|   - Fast lookup by key (O(1) average)                   |
|   - Keys must be hashable (strings, numbers, tuples)    |
|   - Values can be any type                              |
|   - Preserves insertion order (Python 3.7+)             |
|                                                         |
+---------------------------------------------------------+
```

### 1.1 Creating Dictionaries

In [ ]:
# Curly-brace literal
gene_lengths = {
    "BRCA1": 81189,
    "TP53":  19149,
    "EGFR":  188307,
    "MYC":   4150,
}
print(f"Gene lengths: {gene_lengths}")

# dict() constructor from keyword arguments
complement = dict(A='T', T='A', G='C', C='G')
print(f"Complement map: {complement}")

# dict() from list of (key, value) pairs
pairs = [("ATG", "Met"), ("TGG", "Trp"), ("TAA", "Stop")]
codon_names = dict(pairs)
print(f"Codon names: {codon_names}")

# Empty dictionary
annotations = {}
print(f"Empty dict: {annotations}")

### 1.2 Accessing Values

In [ ]:
gene_lengths = {"BRCA1": 81189, "TP53": 19149, "EGFR": 188307, "MYC": 4150}

# Direct access -- raises KeyError if key is missing
print(f"BRCA1 length: {gene_lengths['BRCA1']} bp")

# Safe access with .get() -- returns None (or a default) if missing
print(f"KRAS length:  {gene_lengths.get('KRAS')}")
print(f"KRAS length:  {gene_lengths.get('KRAS', 'not found')}")

# Membership test
print(f"\n'TP53' in dict: {'TP53' in gene_lengths}")
print(f"'KRAS' in dict: {'KRAS' in gene_lengths}")

### 1.3 Modifying Dictionaries

In [ ]:
gene_info = {"BRCA1": 81189, "TP53": 19149}

# Add a new entry
gene_info["EGFR"] = 188307
print(f"After add:    {gene_info}")

# Update an existing value
gene_info["TP53"] = 19200  # corrected length
print(f"After update: {gene_info}")

# Update multiple entries at once
gene_info.update({"MYC": 4150, "KRAS": 45693})
print(f"After batch:  {gene_info}")

# Remove an entry
removed_val = gene_info.pop("KRAS")
print(f"Removed KRAS ({removed_val} bp): {gene_info}")

# setdefault() -- only set if key is absent
gene_info.setdefault("TP53", 0)   # TP53 exists, so no change
gene_info.setdefault("RB1", 180388)  # RB1 is new
print(f"After setdefault: {gene_info}")

### 1.4 Dictionary Iteration

In [ ]:
gene_data = {
    "BRCA1": {"chr": "17", "length": 81189, "gc": 42.5},
    "TP53":  {"chr": "17", "length": 19149, "gc": 48.7},
    "EGFR":  {"chr": "7",  "length": 188307, "gc": 55.1},
}

# Iterate over keys (default)
print("Gene names:", list(gene_data.keys()))

# Iterate over values
total_length = sum(info["length"] for info in gene_data.values())
print(f"Total length: {total_length:,} bp")

# Iterate over key-value pairs
print(f"\n{'Gene':<8} {'Chr':<6} {'Length':>10} {'GC%':>6}")
print("-" * 34)
for gene, info in gene_data.items():
    print(f"{gene:<8} chr{info['chr']:<4} {info['length']:>9,} {info['gc']:>5.1f}%")

### 1.5 The Genetic Code as a Dictionary

The codon table is the quintessential biological dictionary: 64 codons map to 20 amino acids + stop signals.

In [ ]:
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

def translate(dna):
    """Translate a DNA coding sequence into a protein string."""
    protein = []
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3]
        aa = GENETIC_CODE.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)

cds = "ATGGCCGATCGATAG"
print(f"DNA:     {cds}")
print(f"Protein: {translate(cds)}")

### 1.6 Nucleotide Frequency Counter

In [ ]:
# Manual counting with a dictionary
sequence = "ATGCGATCGATCGTAGCGATCGATCGATGCGA"

freq = {}
for nt in sequence:
    freq[nt] = freq.get(nt, 0) + 1

print(f"Sequence ({len(sequence)} bp): {sequence}")
print("\nNucleotide frequencies:")
for nt in 'ATGC':
    count = freq.get(nt, 0)
    pct = count / len(sequence) * 100
    bar = '#' * count
    print(f"  {nt}: {count:>3} ({pct:5.1f}%)  {bar}")

gc_pct = (freq.get('G', 0) + freq.get('C', 0)) / len(sequence) * 100
print(f"\nGC content: {gc_pct:.1f}%")

### 1.7 Nested Dictionaries -- Gene Annotation Database

In [ ]:
gene_annotations = {
    "BRCA1": {
        "full_name": "breast cancer type 1 susceptibility protein",
        "chromosome": "17",
        "coordinates": (43044295, 43125483),
        "strand": "-",
        "go_terms": ["DNA repair", "tumor suppression", "transcription regulation"],
        "diseases": ["Breast cancer", "Ovarian cancer"],
    },
    "TP53": {
        "full_name": "tumor protein p53",
        "chromosome": "17",
        "coordinates": (7661779, 7687538),
        "strand": "-",
        "go_terms": ["apoptosis", "cell cycle arrest", "DNA repair"],
        "diseases": ["Li-Fraumeni syndrome", "Most cancers"],
    },
    "EGFR": {
        "full_name": "epidermal growth factor receptor",
        "chromosome": "7",
        "coordinates": (55019017, 55207337),
        "strand": "+",
        "go_terms": ["signal transduction", "cell proliferation"],
        "diseases": ["Non-small cell lung cancer", "Glioblastoma"],
    },
}

# Query the annotation database
for gene_id, info in gene_annotations.items():
    start, end = info["coordinates"]
    print(f"{gene_id} ({info['full_name']})")
    print(f"  chr{info['chromosome']}:{start:,}-{end:,} ({info['strand']})")
    print(f"  GO terms: {', '.join(info['go_terms'])}")
    print()

### 1.8 defaultdict and Counter

In [ ]:
from collections import defaultdict

# defaultdict automatically creates a default value for missing keys
# Example: group genes by chromosome
gene_locations = [
    ("BRCA1", "chr17"),
    ("TP53",  "chr17"),
    ("EGFR",  "chr7"),
    ("MYC",   "chr8"),
    ("KRAS",  "chr12"),
    ("RB1",   "chr13"),
    ("BRAF",  "chr7"),
]

# Without defaultdict you need: if chrom not in by_chrom: by_chrom[chrom] = []
by_chrom = defaultdict(list)
for gene, chrom in gene_locations:
    by_chrom[chrom].append(gene)

print("Genes grouped by chromosome:")
for chrom in sorted(by_chrom):
    print(f"  {chrom}: {by_chrom[chrom]}")

In [ ]:
from collections import Counter

# Counter -- specialized dict for counting
sequence = "ATGCGATCGATCGATCGATGCGATCG"

# Count nucleotides
nt_counts = Counter(sequence)
print(f"Nucleotide counts: {nt_counts}")
print(f"Most common: {nt_counts.most_common(2)}")

# Count k-mers
k = 3
kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
kmer_counts = Counter(kmers)

print(f"\n{k}-mer frequencies:")
for kmer, count in kmer_counts.most_common(5):
    print(f"  {kmer}: {count}")

In [ ]:
# Counter arithmetic -- compare k-mer profiles between two sequences
from collections import Counter

seq_a = "ATGATGATGCCC"
seq_b = "ATGATGGGGATG"

kmers_a = Counter(seq_a[i:i+3] for i in range(len(seq_a) - 2))
kmers_b = Counter(seq_b[i:i+3] for i in range(len(seq_b) - 2))

print(f"Seq A k-mers: {dict(kmers_a)}")
print(f"Seq B k-mers: {dict(kmers_b)}")

# Common k-mers (minimum of each count)
common = kmers_a & kmers_b
print(f"\nShared k-mers (min counts): {dict(common)}")

# Combined k-mers
combined = kmers_a + kmers_b
print(f"Combined k-mers (sum):      {dict(combined.most_common(5))}")

---

## 2. Sets -- Collections of Unique Elements

A set stores **unique, unordered** elements. Membership testing (`x in my_set`) is O(1), making sets perfect for checking whether something has been seen before.

```
+---------------------------------------------------------+
|                     SET STRUCTURE                        |
+---------------------------------------------------------+
|                                                         |
|   nucleotides = {'A', 'T', 'G', 'C'}                   |
|                                                         |
|   - Unordered (no index access)                         |
|   - Only unique elements (duplicates ignored)           |
|   - Fast membership testing O(1)                        |
|   - Supports union, intersection, difference            |
|                                                         |
+---------------------------------------------------------+
```

### 2.1 Creating Sets

In [ ]:
# From a literal
dna_bases = {'A', 'T', 'G', 'C'}
rna_bases = {'A', 'U', 'G', 'C'}

# From a sequence -- automatically removes duplicates
seq = "ATGCGATCGATCGATCGATCG"
unique_nt = set(seq)
print(f"Unique nucleotides in '{seq}': {unique_nt}")

# Empty set -- must use set(), NOT {} (that creates an empty dict!)
empty_set = set()
print(f"Empty set: {empty_set}, type: {type(empty_set)}")

# From a list of k-mers
k = 3
unique_kmers = set(seq[i:i+k] for i in range(len(seq) - k + 1))
print(f"Unique {k}-mers: {unique_kmers}")

### 2.2 Set Operations

```
+---------------------------------------------------------+
|                   SET OPERATIONS                         |
+---------------------------------------------------------+
|                                                         |
|   A = {1, 2, 3}       B = {2, 3, 4}                    |
|                                                         |
|   Union        (A | B):  {1, 2, 3, 4}                  |
|   Intersection (A & B):  {2, 3}                        |
|   Difference   (A - B):  {1}                           |
|   Sym. Diff    (A ^ B):  {1, 4}                        |
|                                                         |
|        +-----+     +-----+                              |
|        |  A  |     |  B  |                              |
|        |  1  | 2 3 |  4  |                              |
|        +-----+     +-----+                              |
|                                                         |
+---------------------------------------------------------+
```

In [ ]:
# Gene sets from two studies
cancer_genes     = {"BRCA1", "TP53", "EGFR", "MYC", "KRAS"}
dna_repair_genes = {"BRCA1", "BRCA2", "ATM", "MLH1", "TP53"}

# Union -- all genes mentioned in either study
all_genes = cancer_genes | dna_repair_genes
print(f"All genes (union):            {all_genes}")

# Intersection -- genes in BOTH sets
shared = cancer_genes & dna_repair_genes
print(f"In both sets (intersection):  {shared}")

# Difference -- cancer genes NOT involved in DNA repair
cancer_only = cancer_genes - dna_repair_genes
print(f"Cancer only (difference):     {cancer_only}")

# Symmetric difference -- in one set but not both
exclusive = cancer_genes ^ dna_repair_genes
print(f"Exclusive to one (sym diff):  {exclusive}")

In [ ]:
# Practical: find shared genes between three organisms
human_genes   = {"TP53", "BRCA1", "EGFR", "MYC", "KRAS", "RB1"}
mouse_genes   = {"Trp53", "Brca1", "Egfr", "Myc", "Kras", "Rb1", "Pax6"}

# Case-insensitive comparison
human_upper = {g.upper() for g in human_genes}
mouse_upper = {g.upper() for g in mouse_genes}

orthologs = human_upper & mouse_upper
print(f"Shared orthologs (case-insensitive): {orthologs}")
print(f"Human-only: {human_upper - mouse_upper}")
print(f"Mouse-only: {mouse_upper - human_upper}")

In [ ]:
# Validate a DNA sequence using sets
def validate_dna(sequence):
    """Check if a sequence contains only valid DNA nucleotides."""
    valid = {'A', 'T', 'G', 'C'}
    found = set(sequence.upper())
    invalid = found - valid
    if invalid:
        return False, f"Invalid characters: {invalid}"
    return True, "Valid DNA"

test_sequences = [
    "ATGCGATCGATCG",
    "ATGCUGATC",      # U is RNA
    "ATGCXNATC",      # X, N are ambiguous
]

for seq in test_sequences:
    ok, msg = validate_dna(seq)
    print(f"  {seq:<18} {msg}")

### 2.3 Set of Unique K-mers

In [ ]:
# Compare k-mer content between two sequences
seq1 = "ATGCGATCGATCG"
seq2 = "ATGCGGGGATCCC"
k = 3

kmers1 = {seq1[i:i+k] for i in range(len(seq1) - k + 1)}
kmers2 = {seq2[i:i+k] for i in range(len(seq2) - k + 1)}

print(f"Seq1 unique {k}-mers: {kmers1}")
print(f"Seq2 unique {k}-mers: {kmers2}")

shared  = kmers1 & kmers2
only_s1 = kmers1 - kmers2
only_s2 = kmers2 - kmers1

print(f"\nShared:       {shared}")
print(f"Only in seq1: {only_s1}")
print(f"Only in seq2: {only_s2}")

# Jaccard similarity (shared / total unique)
jaccard = len(shared) / len(kmers1 | kmers2)
print(f"\nJaccard similarity: {jaccard:.3f}")

### 2.4 Frozenset -- Immutable Sets

A `frozenset` is a set that cannot be modified after creation. Because it is hashable, it can be used as a dictionary key or as an element of another set.

In [ ]:
# frozenset as a dictionary key -- map codon groups to amino acids
# Each amino acid is encoded by a frozen set of codons
aa_codons = {
    frozenset({"GCT", "GCC", "GCA", "GCG"}): "Ala",
    frozenset({"TGT", "TGC"}):                "Cys",
    frozenset({"GAT", "GAC"}):                "Asp",
    frozenset({"ATG"}):                        "Met",
    frozenset({"TGG"}):                        "Trp",
}

# Look up which amino acid a codon set encodes
for codons, aa in aa_codons.items():
    print(f"{aa}: {len(codons)} codon(s) -- {', '.join(sorted(codons))}")

# frozenset is immutable
fs = frozenset({"ATG", "GTG"})
# fs.add("TTG")  # AttributeError!

---

## 3. Combining Dictionaries and Sets

### 3.1 Reverse Mapping: Amino Acid to Codons

In [ ]:
from collections import defaultdict

# Build reverse codon table: amino acid -> set of codons
reverse_code = defaultdict(set)
for codon, aa in GENETIC_CODE.items():
    reverse_code[aa].add(codon)

print(f"{'AA':<4} {'#Codons':>7}   Codons")
print("-" * 55)
for aa in sorted(reverse_code):
    codons = sorted(reverse_code[aa])
    print(f"{aa:<4} {len(codons):>7}   {', '.join(codons)}")

### 3.2 Gene Pathway Analysis

In [ ]:
# Pathways as dict of sets
pathways = {
    "DNA_repair":     {"BRCA1", "BRCA2", "ATM", "TP53", "MLH1"},
    "Cell_cycle":     {"TP53", "RB1", "CDK4", "CCND1", "MYC"},
    "Apoptosis":      {"TP53", "BCL2", "BAX", "CASP3", "FAS"},
    "Signal_transduction": {"EGFR", "KRAS", "BRAF", "MEK1", "ERK2"},
}

# Which pathways does TP53 participate in?
query_gene = "TP53"
tp53_pathways = [name for name, genes in pathways.items() if query_gene in genes]
print(f"{query_gene} is in pathways: {tp53_pathways}")

# Genes shared between DNA_repair and Cell_cycle
shared = pathways["DNA_repair"] & pathways["Cell_cycle"]
print(f"\nDNA_repair AND Cell_cycle: {shared}")

# Genes unique to Apoptosis (not in any other pathway)
other_genes = set()
for name, genes in pathways.items():
    if name != "Apoptosis":
        other_genes |= genes

apoptosis_unique = pathways["Apoptosis"] - other_genes
print(f"Unique to Apoptosis: {apoptosis_unique}")

---

## 4. Exercises

### Exercise 1: Build a Codon-to-Amino-Acid Table

Write a function that, given a list of `(codon, amino_acid)` pairs, builds a dictionary mapping each codon to its amino acid. Then use it to translate a short coding sequence.

In [ ]:
# Exercise 1: Build codon table and translate

def build_codon_table(pairs):
    """
    Build a codon -> amino acid dictionary.
    
    Parameters:
        pairs: list of (codon, amino_acid) tuples
    Returns:
        dict mapping codon string to amino acid character
    """
    # YOUR CODE HERE
    pass

def translate_with_table(dna, codon_table):
    """
    Translate DNA to protein using the given codon table.
    Stop at the first stop codon ('*').
    """
    # YOUR CODE HERE
    pass

# Test data
# pairs = [("ATG", "M"), ("GCC", "A"), ("GAT", "D"), ("CGA", "R"), ("TAG", "*"), ...]
# translate_with_table("ATGGCCGATCGATAG", table) -> "MADR"

In [ ]:
# --- Solution ---

def build_codon_table(pairs):
    return {codon: aa for codon, aa in pairs}

def translate_with_table(dna, codon_table):
    protein = []
    for i in range(0, len(dna) - 2, 3):
        codon = dna[i:i+3]
        aa = codon_table.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)

# Build table from the full GENETIC_CODE
pairs = list(GENETIC_CODE.items())
my_table = build_codon_table(pairs)

test_cds = "ATGGCCGATCGATAG"
result = translate_with_table(test_cds, my_table)
print(f"DNA:     {test_cds}")
print(f"Protein: {result}")

### Exercise 2: Find Shared Genes Between Organisms

Given gene sets from three organisms, find:
1. Genes present in all three (core genes)
2. Genes present in exactly two organisms
3. Genes unique to each organism

In [ ]:
# Exercise 2: Shared genes between organisms

human = {"TP53", "BRCA1", "EGFR", "MYC", "KRAS", "RB1", "PTEN"}
mouse = {"TP53", "BRCA1", "EGFR", "MYC", "KRAS", "PAX6", "SOX2"}
zebrafish = {"TP53", "BRCA1", "MYC", "PAX6", "FGF8", "SHH"}

# YOUR CODE HERE:
# 1. Find genes in ALL three organisms
# 2. Find genes in exactly two organisms
# 3. Find genes unique to each organism

In [ ]:
# --- Solution ---

human = {"TP53", "BRCA1", "EGFR", "MYC", "KRAS", "RB1", "PTEN"}
mouse = {"TP53", "BRCA1", "EGFR", "MYC", "KRAS", "PAX6", "SOX2"}
zebrafish = {"TP53", "BRCA1", "MYC", "PAX6", "FGF8", "SHH"}

# 1. Core genes (in all three)
core = human & mouse & zebrafish
print(f"Core genes (all 3): {core}")

# 2. Genes in exactly two organisms
all_organisms = {"human": human, "mouse": mouse, "zebrafish": zebrafish}
all_genes = human | mouse | zebrafish

in_exactly_two = set()
for gene in all_genes:
    count = sum(1 for org_genes in all_organisms.values() if gene in org_genes)
    if count == 2:
        in_exactly_two.add(gene)
print(f"In exactly 2: {in_exactly_two}")

# 3. Unique to each organism
for name, genes in all_organisms.items():
    others = set()
    for other_name, other_genes in all_organisms.items():
        if other_name != name:
            others |= other_genes
    unique = genes - others
    print(f"Unique to {name}: {unique}")

### Exercise 3: Amino Acid Composition Counter

Write a function that translates a DNA sequence and then counts the frequency of each amino acid in the resulting protein. Return a dictionary sorted by frequency.

In [ ]:
# Exercise 3: Amino acid composition

def amino_acid_composition(dna_sequence):
    """
    Translate DNA and count amino acid frequencies.
    
    Parameters:
        dna_sequence (str): DNA coding sequence
    
    Returns:
        list of (amino_acid, count) tuples sorted by count descending
    """
    # YOUR CODE HERE
    pass

# Test
# cds = "ATGGCCGATCGAGCCGCCATGGCCGATCGA"
# amino_acid_composition(cds) should return something like:
# [('A', 4), ('D', 2), ('R', 2), ('M', 2), ...]

In [ ]:
# --- Solution ---

from collections import Counter

def amino_acid_composition(dna_sequence):
    protein = translate(dna_sequence)
    counts = Counter(protein)
    return counts.most_common()

cds = "ATGGCCGATCGAGCCGCCATGGCCGATCGA"
protein = translate(cds)
composition = amino_acid_composition(cds)

print(f"DNA:     {cds}")
print(f"Protein: {protein}")
print(f"\nAmino acid composition:")
for aa, count in composition:
    bar = '#' * (count * 3)
    print(f"  {aa}: {count}  {bar}")

### Exercise 4: Codon Usage Bias

For each amino acid, calculate which synonymous codon is used most frequently in a given CDS. This is called **codon usage bias** and varies between organisms.

In [ ]:
# Exercise 4: Codon usage bias

def codon_usage_bias(dna_sequence):
    """
    Analyze codon usage bias: for each amino acid, show
    which synonymous codons are used and their frequencies.
    
    Parameters:
        dna_sequence (str): DNA coding sequence
    
    Returns:
        dict: {amino_acid: {codon: count, ...}, ...}
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Solution ---

from collections import defaultdict, Counter

def codon_usage_bias(dna_sequence):
    codons = [dna_sequence[i:i+3] for i in range(0, len(dna_sequence) - len(dna_sequence) % 3, 3)]
    
    usage = defaultdict(Counter)
    for codon in codons:
        aa = GENETIC_CODE.get(codon, 'X')
        usage[aa][codon] += 1
    
    return dict(usage)

# Test with a longer CDS
cds = "ATGGCCGCCGCTGATGACGAAGAGGCAGCCGCTATGGCCATAGCGGATGATTAG"
bias = codon_usage_bias(cds)

print(f"CDS: {cds}\n")
for aa in sorted(bias):
    codons = bias[aa]
    total = sum(codons.values())
    codon_str = ", ".join(f"{c}:{n}" for c, n in codons.most_common())
    print(f"  {aa} (total {total}): {codon_str}")

---

## Key Takeaways

1. **Dictionaries** provide O(1) key lookup -- ideal for codon tables, gene annotations, and any ID-to-data mapping.
2. **`defaultdict`** eliminates the need to check for missing keys before appending or counting.
3. **`Counter`** is purpose-built for frequency counting (nucleotides, k-mers, amino acids).
4. **Sets** give you fast membership testing and powerful operations (union, intersection, difference) for comparing gene lists.
5. **`frozenset`** is the immutable, hashable version of a set -- useful as dictionary keys.
6. **Nested dicts** model complex biological data (gene annotations with coordinates, GO terms, diseases).

---

### Next: Module 10 -- Comprehensions

---

[← Previous: Module 8: Lists and Tuples](../../Tier_1_Python_for_Bioinformatics/08_Lists_and_Tuples/01_lists_and_tuples.ipynb) | [Next: Module 10: Comprehensions →](../../Tier_1_Python_for_Bioinformatics/10_Comprehensions/01_comprehensions.ipynb)